In [95]:
import pandas as pd
import re

# Everest Death Data from Wikipedia

Source: https://en.wikipedia.org/wiki/List_of_people_who_died_climbing_Mount_Everest


In [96]:
#import data
ever_death_wiki = pd.read_csv("data/everest_death_wikipedia.csv")
ever_death_wiki.head()

,Unnamed: 0,Date,Age,Expedition,Nationality,Cause of death,Location,Remains status,Refs
0,Alexander Mitchell Kellas,5-Jun-21,52,1921 British Mount Everest reconnaissance expe...,United Kingdom,Heart attack,En route to base camp near the Tibetan village...,"Recovered, buried near Kampa Dzong",[9][26]
1,Unknown porter,5-Jun-21,NaN,1921 British Mount Everest reconnaissance expe...,NaN,NaN,En route to base camp near the Tibetan village...,NaN,[10]
2,Dorje,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9]
3,Lhakpa,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9]
4,Norbu,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9]


In [97]:
ever_death_wiki.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Unnamed: 0      347 non-null    object
 1   Date            346 non-null    object
 2   Age             214 non-null    object
 3   Expedition      311 non-null    object
 4   Nationality     346 non-null    object
 5   Cause of death  336 non-null    object
 6   Location        329 non-null    object
 7   Remains status  99 non-null     object
 8   Refs            346 non-null    object
dtypes: object(9)
memory usage: 24.7+ KB


In [98]:
#how many NANs in "Location" column?
ever_death_wiki["Location"].isna().sum()

#save to no_location df
no_location = ever_death_wiki[ever_death_wiki["Location"].isna()]
no_location.head()

,Unnamed: 0,Date,Age,Expedition,Nationality,Cause of death,Location,Remains status,Refs
15,Wang Ji,11-Apr-60,NaN,Chinese Expedition Northern Slope,China,Mountain sickness,NaN,NaN,[32][33]
21,Kunga Norbu,5-Apr-70,NaN,Japanese Skiing Expedition,Nepal,Avalanche,NaN,NaN,[40]
107,Ang Pinjo,12-Dec-89,NaN,South Korean expedition,Nepal,"Altitude sickness, heart attack",NaN,NaN,[38]
108,Rafael G—mez-Menor,14-Sep-90,NaN,NaN,Spain,Avalanche,NaN,NaN,[84]
109,Ang Sona,14-Sep-90,NaN,NaN,Nepal,Avalanche,NaN,NaN,[84]


In [99]:
#remove locations = NAN
ever_death_wiki = ever_death_wiki.dropna(subset=["Location"])
ever_death_wiki.head()
#if Nationality contains NAN, replace with "Unknown"
ever_death_wiki["Nationality"] = ever_death_wiki["Nationality"].fillna("Unknown")
ever_death_wiki.head()


#create Role Column - if Nationality contains "Nepal", Role = "Nepalese", else "Climber" 
ever_death_wiki["Role"] = ever_death_wiki["Nationality"].apply(lambda x: "Nepalese" if "Nepal" in x else "Climber")


#change name of "Unnamed: 0" column to "Name"
ever_death_wiki = ever_death_wiki.rename(columns={"Unnamed: 0": "Name"})
ever_death_wiki.head() 

,Name,Date,Age,Expedition,Nationality,Cause of death,Location,Remains status,Refs,Role
0,Alexander Mitchell Kellas,5-Jun-21,52,1921 British Mount Everest reconnaissance expe...,United Kingdom,Heart attack,En route to base camp near the Tibetan village...,"Recovered, buried near Kampa Dzong",[9][26],Climber
1,Unknown porter,5-Jun-21,NaN,1921 British Mount Everest reconnaissance expe...,Unknown,NaN,En route to base camp near the Tibetan village...,NaN,[10],Climber
2,Dorje,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9],Nepalese
3,Lhakpa,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9],Nepalese
4,Norbu,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9],Nepalese


In [100]:
#sort location by most common
pd.set_option('display.max_rows', None)

ever_death_wiki["Location"].value_counts()

Location
Icefall                                                                 36
Base Camp                                                               27
Below North Col                                                         13
8600m N.E. Ridge                                                        10
Camp II                                                                  7
Near Summit                                                              7
N.E. Ridge                                                               7
Balcony                                                                  7
South Col                                                                7
6400m                                                                    7
8500m N.E. Ridge                                                         5
Camp IV                                                                  5
8700m N.E. Ridge                                                         5
South Summit    

In [101]:
#Verifying "Base Camp" deaths
    #people with "Base Camp" in "Location" col of ever_death_wiki
ever_death_wiki[ever_death_wiki["Location"].str.contains("Base Camp", na=False)]



,Name,Date,Age,Expedition,Nationality,Cause of death,Location,Remains status,Refs,Role
38,Mingma ; deaf-mute Sherpa porter,Aug-75,"""Young""",1975 British Mount Everest Southwest Face expe...,Nepal,Drowned,Below Base Camp,NaN,[48],Nepalese
150,Malcolm Duff,23-Apr-97,43,British,United Kingdom,Heart attack,Base Camp,NaN,[38][94],Climber
222,Shailendra Kumar Upadhyaya,9-May-11,82,Nepal,Nepal,Altitude sickness,Base Camp,NaN,[142],Nepalese
225,Hiroaki Kino,15-Sep-11,48,Bochi Bochi Trek,Japan,Cerebral apoplexy,Base Camp,NaN,[145][146][147],Climber
227,Karsang Namgyal Sherpa,19-Apr-12,NaN,Prestige Adventures,Nepal,Altitude sickness,Base Camp,NaN,[149][150],Nepalese
228,Ramesh Gulve,20-Apr-12,33,Pune team,India,Stroke,Base Camp,NaN,[149][151],Climber
247,Mingma Tenzing Sherpa,2-Apr-14,NaN,Peak Freaks Expedition,Nepal,HAPE,Base Camp,NaN,[167],Nepalese
263,Daniel Paul Fredinburg,25-Apr-15,33,2015 Jagged Globe Everest Expedition,United States,Base Camp avalanche following the April 2015 N...,Base Camp,Recovered,[16][15][17][168][169][170][171][172][173],Climber
264,Marisa Eve Girawong,25-Apr-15,28,H.G. Nuptse Expedition 2015,United States,Base Camp avalanche following the April 2015 N...,Base Camp,Recovered,[169][170][171][172][173],Climber
265,Thomas Ely Taplin,25-Apr-15,61,TET Films & Photography,United States,Base Camp avalanche following the April 2015 N...,Base Camp,NaN,[170][174],Climber


In [131]:
ever_death_cleaned_locations = ever_death_wiki.copy()

#remove commas from "Location" column
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].str.replace(",", "")

#all deaths that mention "Balcony" in the location will be categorized as "Balcony"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Balcony" if re.search("Balcony", x) else x)
#all deaths mentioning Hillary Step will be categorized as "Hillary Step"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Hillary Step" if re.search("Hillary Step", x) else x)
#Hornbein Couloir will be categorized as "Hornbein Couloir"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Hornbein Couloir" if re.search("Hornbein Couloir", x) else x)
#all icefall becomes Khumbu Icefall
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Khumbu Icefall" if re.search("Icefall", x) else x)


#Routes should be organized from bottom to top -> need to parse out North Route, South Route, and West Ridge route
    #because some locations are names and some are elevations, will add evelations to front of location name

#if "Base Camp North Side" in location, add "6490m" to front of location name
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "6490m " + x if re.search("Base Camp North Side", x) else x)
# if "North Col" in location, add "7000m" to front of location name
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "7000m " + x if re.search("North Col", x) else x)
# if "Karl" in "Name", change location to 6400m Below North Col
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "6400m Below North Col" if re.search("Karl", row["Name"]) else row["Location"], axis=1)
#if only 6400m in location, change to 6400m W Ridge
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "6400m W Ridge" if re.search("6400m", x) and not re.search("North Col", x) else x)
#if "Above" or "Descending" and "South Col" in location, change to "8000m Above South Col"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "8000m Above South Col" if (re.search("Above", row["Location"]) or re.search("Descending", row["Location"])) and re.search("South Col", row["Location"]) else row["Location"], axis=1)
#if "Above South Col" and no numeric characters in location, change to "8000m Above South Col"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "8000m Above South Col" if re.search("Above South Col", row["Location"]) and not re.search("\d", row["Location"]) else row["Location"], axis=1)
#locations where "South Col" appears with no numeric characters, add 7900m to front of location name
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "7900m " + row["Location"] if re.search("South Col", row["Location"]) and not re.search("\d", row["Location"]) else row["Location"], axis=1)
#If Marty Hoey is in "Name", change location to "8000m N.E. Ridge"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "8000m N.E. Ridge" if re.search("Marty Hoey", row["Name"]) else row["Location"], axis=1)
#if only "8000m" in location, change to "8000m W Ridge"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "8000m W Ridge" if re.search("8000m", x) and not re.search("South Col", x) and not re.search("N.E. Ridge", x) else x)
#if "Lhotse Face" or "Lhoste face" in location, change to "7400m Lhotse Face"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "7400m Lhotse Face" if re.search("Lhotse Face", x) or re.search("Lhotse face", x) else x)
#if only "Near Camp IV" in location, change to "Camp IV"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Camp IV" if re.search("Near Camp IV", x) else x)
#if "Camp IV (south side) in tent" or "Camp IV in the tent" in location, change to "Camp IV"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Camp IV" if re.search("Camp IV \(south side\) in tent", x) or re.search("Camp IV in the tent", x) else x)
#if only "Base camp" case senitive, change to "Base Camp"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Base Camp" if re.search("Base camp", x) else x)
#if contains "Base Camp North Side" or "Rongbuk", chang to "Base Camp North Side"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Base Camp North Side" if re.search("Base Camp North Side", x) or re.search("Rongbuk", x) else x)
#if "NE" in location, change that text to "N.E." same with SE to S.E.
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: re.sub(r"\bNE\b", "N.E.", x))
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: re.sub(r"\bSE\b", "S.E.", x))
#if "S.W. Face" in location and no numeric, change to "7700m S.W. Face"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "7700m S.W. Face" if re.search("S.W. Face", row["Location"]) and not re.search("\d", row["Location"]) else row["Location"], axis=1)
#if "W ridge" or "W Ridge" in location, repalce with "West Ridge"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: re.sub(r"W Ridge|W ridge", "West Ridge", x))
#"Near summit" and "Below the Summit" becomes "Near Summit"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Near Summit" if re.search("Near summit", x) or re.search("Below the Summit", x) else x)
#if "6900m" appears, change to "6900m W. Cwm"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "6900m W. Cwm" if re.search("6900m", x) else x)
#"North-East Ridge (approx. 8200m)" to "8200m N.E. Ridge"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "8200m N.E. Ridge" if re.search("North-East Ridge", x) and re.search("8200m", x) else x)


ever_death_cleaned_locations["Location"].value_counts()


Location
Khumbu Icefall                                                          46
Base Camp                                                               30
7000m Below North Col                                                   13
Balcony                                                                 13
Near Summit                                                             10
Camp IV                                                                 10
8600m N.E. Ridge                                                        10
7400m Lhotse Face                                                        8
7900m South Col                                                          8
Hillary Step                                                             8
8000m West Ridge                                                         7
N.E. Ridge                                                               7
Camp II                                                                  7
6400m West Ridge

In [132]:
#adding a new "Route" column to categorize locations into North Route, South Route, and West Ridge
def categorize_route(location):
    #if "North" or "N.E." in location, return "North"
    if re.search("North", location) or re.search("N\.E\.", location) or re.search("Hillary Step", location) or re.search("Hornbein Couloir", location) or re.search("Norton Couloir", location) or re.search("north", location) or re.search("Tibet", location):
        return "North"
    #if "W." or "S.W." or "Lhola camp" or "West Ridge" in location, return "West Ridge" 
    elif re.search("S\.W\.", location) or re.search("Lhola camp", location) or re.search("West Ridge", location):
        return "West Ridge"
    #if "South", "S.E.", "Kumbu Icefall", "Base Camp" with nothing else, return "South"
    #added balcony and Lhotse
    elif re.search("South", location) or re.search("Cwm", location) or re.search("south", location) or re.search("Lhotse", location) or re.search("Balcony", location) or re.search("S\.E\.", location) or re.search("Khumbu Icefall", location) or (re.search("Base Camp", location) and not re.search("North", location)):
        return "South"
    #else return "Other"
    else:
        return "Other"
    
ever_death_cleaned_locations["Route"] = ever_death_cleaned_locations["Location"].apply(categorize_route)

#if in "Other" and "Camp" in location, change route to "South"
ever_death_cleaned_locations["Route"] = ever_death_cleaned_locations.apply(lambda row: "South" if row["Route"] == "Other" and re.search("Camp", row["Location"]) else row["Route"], axis=1)

In [133]:
ever_death_cleaned_locations.groupby("Route")["Location"].value_counts()

Route       Location                                                            
North       7000m Below North Col                                                   13
            8600m N.E. Ridge                                                        10
            Hillary Step                                                             8
            N.E. Ridge                                                               7
            8500m N.E. Ridge                                                         5
            8700m N.E. Ridge                                                         5
            Base Camp North Side                                                     5
            7800m N.E. Ridge                                                         3
            8200m N.E. Ridge                                                         3
            8300m N.E. Ridge                                                         3
            8750m N.E. Ridge                     

In [ ]:
#locations where "South Col" appears with no numeric characters
ever_death_cleaned_locations[ever_death_cleaned_locations["Location"].str.contains("South Col") & ~ever_death_cleaned_locations["Location"].str.contains("\d")]["Location"].value_counts()

In [ ]:
#look at all "Location" that start with a non numeric character
ever_death_cleaned_locations[~ever_death_cleaned_locations["Location"].str.match("^\d")]["Location"].value_counts()

In [ ]:
ever_death_cleaned_locations["Location"].value_counts()

#locations grouped by nationality
ever_death_cleaned_locations.groupby("Role")["Location"].value_counts()



## Everest Expedition Data for Routes

In [ ]:
routes = pd.read_csv("data/EverestExpeditions_HimalayanDatabase.csv")
routes.head()


In [ ]:
routes.info()

In [ ]:
routes.iloc[3,7]

In [ ]:
#column names?
routes.columns
#remove spaces from column names
routes.columns = routes.columns.str.replace(" ", "")
routes.columns
#for "Route(s)", replace / with AND-OR
routes["Route(s)"] = routes["Route(s)"].str.replace("/", " AND-OR ")


In [ ]:
#drop rows where "Dead" is nan
routes = routes.dropna(subset=["Dead"])
routes.head(10)

In [ ]:
#count number of instances of each unique route in "Route(s)"
routes["Route(s)"].value_counts()


In [ ]:
#number of unique routes and number of deaths for each route
route_deaths = routes.groupby("Route(s)")["Dead"].sum()
route_deaths.sort_values(ascending=False)

In [ ]:
#total deaths
total_deaths = routes["Dead"].sum()
print(total_deaths)